In [4]:
import tensorflow as tf
from tensorflow import keras
import os
import json
from pathlib import Path

In [2]:
# ------------------------------------------------------------------
# 1.1 Paths and settings
# ------------------------------------------------------------------
train_dir = "/Users/sanjib700/Desktop/My_Projects/Covide_image/Covid19-dataset/train"
test_dir  = "/Users/sanjib700/Desktop/My_Projects/Covide_image/Covid19-dataset/test"

IMG_SIZE = (150, 150)   # every image gets resized to this, regardless of its original size
BATCH_SIZE = 16         # number of images grouped together per training step

# ------------------------------------------------------------------
# 1.2 Load images directly from folders
#     Keras automatically uses each subfolder name as the class label
# ------------------------------------------------------------------
train_ds_raw = keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",   # one-hot labels, since we have 3 classes
    shuffle=True,
    seed=42,                    # fixed seed -> reproducible shuffle order
)

test_ds_raw = keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,               # no need to shuffle test data
)

class_names = train_ds_raw.class_names
num_classes = len(class_names)
print("Classes found:", class_names)

# ------------------------------------------------------------------
# 1.3 Preprocessing: rescale pixel values from 0-255 down to 0-1
#     Neural networks train more stably on small, consistent input ranges.
# ------------------------------------------------------------------
normalization_layer = keras.layers.Rescaling(1.0 / 255)

train_ds = train_ds_raw.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds_raw.map(lambda x, y: (normalization_layer(x), y))

# ------------------------------------------------------------------
# 1.4 Performance: cache + prefetch so training doesn't wait on disk I/O
#     cache()    -> keeps loaded/processed images in memory after the first epoch
#     prefetch() -> prepares the next batch while the current one is training
# ------------------------------------------------------------------
train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(tf.data.AUTOTUNE)

# ------------------------------------------------------------------
# 1.5 Sanity check: confirm shapes and pixel range before moving on
# ------------------------------------------------------------------
for images, labels in train_ds.take(1):
    print("\nOne batch of images shape:", images.shape)   # (batch, height, width, channels)
    print("One batch of labels shape:", labels.shape)     # (batch, num_classes)
    print("Pixel value range after scaling:", images.numpy().min(), "to", images.numpy().max())

Found 251 files belonging to 3 classes.


2026-09-24 04:07:53.479746: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-09-24 04:07:53.481014: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-09-24 04:07:53.481521: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-09-24 04:07:53.482181: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-09-24 04:07:53.483592: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Found 66 files belonging to 3 classes.
Classes found: ['Covid', 'Normal', 'Viral Pneumonia']

One batch of images shape: (16, 150, 150, 3)
One batch of labels shape: (16, 3)
Pixel value range after scaling: 0.0 to 1.0


2026-09-24 04:07:54.206115: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2026-09-24 04:07:54.233121: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Data Pipeline & Preprocessing — Interpretation

- **Images are loaded directly from folder structure**, using
  `image_dataset_from_directory` — each subfolder name (`Covid`, `Normal`,
  `Viral Pneumonia`) is automatically used as the class label, removing the need to
  build a manual label array.

- **All images are resized to a fixed 150×150 pixels.** Source X-ray images vary in
  original resolution, but every instance fed into a neural network must have an
  identical shape — resizing enforces this consistently across the dataset.

- **Images are grouped into batches of 16** rather than processed one at a time. This
  balances memory usage against training speed and is a standard default for a
  dataset of this size (251 training images).

- **`label_mode="categorical"` one-hot encodes the 3 classes** (e.g. Covid → `[1,0,0]`),
  matching the `categorical_crossentropy` loss function used during training.

- **Training data is shuffled (`shuffle=True`, fixed `seed=42`)** so the model doesn't
  learn from images in a fixed, potentially biased order (e.g. all Covid images
  first). The fixed seed makes the shuffle reproducible across runs. Test data is
  **not** shuffled, since order doesn't matter for evaluation.

- **Pixel values are rescaled from the raw 0–255 range down to 0–1** using a
  `Rescaling(1.0/255)` layer, applied identically to both training and test sets.
  Small, consistent input ranges help gradient descent train more stably.

- **`.cache()` and `.prefetch()` are applied for performance** — `.cache()` keeps
  processed images in memory after the first epoch instead of re-reading from disk
  every time; `.prefetch()` prepares the next batch while the current one is still
  training. Neither changes model results — both are purely speed optimizations.

- **A sanity check confirms the pipeline before modeling begins:** correct number of
  classes detected, expected batch shapes `(16, 150, 150, 3)` for images and
  `(16, 3)` for one-hot labels, and pixel values confirmed in the `[0.0, 1.0]` range
  post-rescaling.

In [7]:
# ==============================================================================
# STEP 2: Dense Model — Build, Train, Evaluate
# ==============================================================================
# Requires Step 1 (data pipeline) to have already run: train_ds, test_ds,
# class_names, num_classes must all exist.

import numpy as np
import json
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix

# ------------------------------------------------------------------
# 2.1 Build the model
#     Flatten + Dense layers only -- no convolutional layers.
#     Baseline: treats every pixel independently, no spatial awareness.
# ------------------------------------------------------------------
dense_model = keras.Sequential([
    keras.layers.Input(shape=(150, 150, 3)),
    keras.layers.Flatten(),
    keras.layers.BatchNormalization(),          # stabilizes the huge flattened input
    keras.layers.Dense(300, activation="relu"),
    keras.layers.BatchNormalization(),          # stabilizes the huge flattened input
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(num_classes, activation="softmax"),
], name="Dense_Baseline")

dense_model.summary()

# ------------------------------------------------------------------
# 2.2 Compile
# ------------------------------------------------------------------
dense_model.compile(
    loss="categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),   # 10x smaller than default
    metrics=["accuracy"],
)

# ------------------------------------------------------------------
# 2.3 Train
#     EarlyStopping monitors validation accuracy, stops if it hasn't improved
#     in 5 epochs, and rolls back to the BEST epoch's weights (not the last one).
# ------------------------------------------------------------------
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True,
)

dense_history = dense_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=30,
    callbacks=[early_stop],
    verbose=2,
)

# Save the REAL training history to disk
Path("Training_history_jason").mkdir(exist_ok=True) # Create the folder
with open("Training_history_jason/dense_history.json", "w") as f:
    json.dump(dense_history.history, f)

# ------------------------------------------------------------------
# 2.4 Evaluate: overall test accuracy
# ------------------------------------------------------------------
test_loss, test_acc = dense_model.evaluate(test_ds, verbose=0)
print(f"\nDense model -- Test accuracy: {test_acc:.4f}")

# ------------------------------------------------------------------
# 2.5 Evaluate: full test set, per-class report + confusion matrix
#     A single accuracy number can hide a model doing well on one class
#     and poorly on another -- this checks every test image individually.
# ------------------------------------------------------------------
all_true = []
all_pred = []

for images, labels in test_ds:
    preds = dense_model.predict(images, verbose=0)
    all_true.extend(labels.numpy().argmax(axis=1))
    all_pred.extend(preds.argmax(axis=1))

all_true = np.array(all_true)
all_pred = np.array(all_pred)

print(f"\nTotal test images checked: {len(all_true)}")

print("\n=== Dense model -- per-class performance ===")
print(classification_report(all_true, all_pred, target_names=class_names))

print("=== Dense model -- confusion matrix ===")
print("Rows = true class, Columns = predicted class")
cm_dense = confusion_matrix(all_true, all_pred)
print("            ", "  ".join(f"{c[:8]:>8s}" for c in class_names))
for i, row in enumerate(cm_dense):
    print(f"{class_names[i][:12]:12s}", "  ".join(f"{v:8d}" for v in row))

Model: "Dense_Baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_2 (Flatten)             │ (None, 67500)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 67500)          │       270,000 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 300)            │    20,250,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 300)            │         1,200 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 3)              │           303 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,551,903 (78.40 MB)

 Trainable params: 20,416,303 (77.88 MB)

 Non-trainable params: 135,600 (529.69 KB)

Epoch 1/30
16/16 - 3s - 176ms/step - accuracy: 0.8446 - loss: 0.8497 - val_accuracy: 0.3939 - val_loss: 6.3833
Epoch 2/30
16/16 - 1s - 54ms/step - accuracy: 0.9482 - loss: 0.1136 - val_accuracy: 0.3939 - val_loss: 3.6598
Epoch 3/30
16/16 - 1s - 49ms/step - accuracy: 0.9841 - loss: 0.0592 - val_accuracy: 0.3939 - val_loss: 2.3289
Epoch 4/30
16/16 - 1s - 53ms/step - accuracy: 0.9960 - loss: 0.0390 - val_accuracy: 0.5152 - val_loss: 1.7051
Epoch 5/30
16/16 - 1s - 51ms/step - accuracy: 1.0000 - loss: 0.0295 - val_accuracy: 0.5455 - val_loss: 1.3089
Epoch 6/30
16/16 - 1s - 50ms/step - accuracy: 1.0000 - loss: 0.0237 - val_accuracy: 0.5909 - val_loss: 1.0723
Epoch 7/30
16/16 - 1s - 50ms/step - accuracy: 1.0000 - loss: 0.0200 - val_accuracy: 0.6515 - val_loss: 0.9251
Epoch 8/30
16/16 - 1s - 53ms/step - accuracy: 1.0000 - loss: 0.0171 - val_accuracy: 0.6818 - val_loss: 0.8226
Epoch 9/30
16/16 - 1s - 56ms/step - accuracy: 1.0000 - loss: 0.0150 - val_accuracy: 0.6970 - val_loss: 0.7479
Epoch 10/

2026-09-24 04:38:44.173731: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Dense Model — Interpretation

- **Test accuracy: 84.85%**, with macro F1 of 0.84 — a strong result for a model with
  no spatial/convolutional structure, and a substantial improvement over the initial
  (pre-BatchNormalization) run, which failed to train (`loss: nan`) due to activation
  explosion in the large flattened input (67,500 values) feeding directly into a
  `Dense(300)` layer.

- **Training accuracy reached 100% by epoch 5** and stayed there for the rest of
  training, while training loss kept shrinking (0.85 → 0.004). This is a classic
  sign of the model fitting the training set very tightly — expected with a small
  dataset (251 images) and a relatively large Dense architecture.

- **Validation accuracy climbed steadily and consistently** (39% → 85%) even after
  training accuracy plateaued at 100%, showing the model continued to generalize
  better on unseen data for many epochs, not just memorize.

- **Validation loss tells a more nuanced story than validation accuracy alone**: it
  dropped sharply through epoch ~16 (down to ~0.56), then began *slowly rising again*
  from epoch 19 onward (0.564 → 0.597) even as accuracy stayed flat or improved
  slightly. This is an early overfitting signal — the model's predictions are
  becoming less confident/calibrated on validation data, even though it's still
  getting the same or more of them right.

- **`EarlyStopping` correctly caught this** — training stopped at epoch 27 (rather
  than running the full 30), and `restore_best_weights=True` rolled the model back
  to its best validation-loss epoch rather than keeping the final, slightly
  overfit epoch's weights.

- **Per-class performance is fairly balanced, with Covid the strongest class**:
  - **Covid**: precision 0.96, recall 0.88 — very few false positives, though 3 of
    26 true Covid cases were missed (2 called Normal, 1 called Viral Pneumonia).
  - **Normal**: precision 0.83, recall 0.75 — the weakest recall of the three;
    5 of 20 true Normal cases were misclassified as Viral Pneumonia.
  - **Viral Pneumonia**: precision 0.75, recall 0.90 — catches most true cases,
    but at the cost of the lowest precision (some Normal cases get pulled in).

- **The confusion matrix shows Normal and Viral Pneumonia are the main source of
  confusion for this model** (5 Normal → Viral Pneumonia, 1 Viral Pneumonia →
  Normal), while Covid is rarely confused with either — consistent with Covid
  presenting more distinctly in this dataset than the Normal/Viral Pneumonia
  distinction does.

- **Overall**: a solid baseline once training stability was fixed, but the
  validation-loss uptick after epoch ~19 suggests this specific run is close to (or
  just past) the point of diminishing returns for this architecture on this dataset
  size — a useful reference point for comparing against the CNN-based models, which
  are expected to generalize better due to their spatial inductive bias.

In [33]:
print(all_true)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]


In [34]:
print(all_pred)

[0 0 0 0 0 0 0 0 0 1 2 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 2 1 1 1 2 2 2 1
 2 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 1 2 2 2 2 2 2 0 2 2 2 2]


In [28]:
for image, lables in test_ds:
    pred = dense_model.predict(image)
    print(pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
[[9.9640036e-01 3.4188062e-03 1.8081852e-04]
 [9.9999821e-01 4.3837272e-08 1.7382323e-06]
 [9.6748936e-01 2.2275511e-02 1.0235074e-02]
 [9.9699688e-01 6.6211132e-06 2.9965148e-03]
 [9.9996042e-01 7.0921385e-07 3.8807124e-05]
 [9.9552625e-01 1.3965842e-03 3.0771140e-03]
 [9.9966097e-01 3.1247223e-04 2.6532010e-05]
 [9.9865389e-01 1.2677021e-03 7.8352939e-05]
 [9.5707953e-01 4.1028652e-02 1.8917986e-03]
 [8.6527456e-05 9.8399377e-01 1.5919747e-02]
 [2.7743585e-02 2.0930730e-01 7.6294911e-01]
 [9.7218293e-01 1.8192774e-02 9.6241832e-03]
 [9.5731276e-01 3.9904330e-02 2.7829842e-03]
 [3.0938023e-01 6.8573606e-01 4.8836209e-03]
 [9.7621095e-01 9.7972772e-04 2.2809273e-02]
 [9.9966884e-01 1.1880578e-04 2.1236416e-04]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
[[9.99196589e-01 3.92691072e-06 7.99499685e-04]
 [9.98807311e-01 1.15968112e-03 3.29782088e-05]
 [9.99439657e-01 4.22261073e-04 1.38124422e-04]
 [9.99445379e-01 4.14838287e-04 1.39711148e-04]
 [9.9966096

2026-09-24 05:12:56.936506: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [38]:
# ==============================================================================
# STEP 3: CNN Model — Build, Train, Evaluate
# ==============================================================================
# Requires Step 1 (data pipeline) to have already run: train_ds, test_ds,
# class_names, num_classes must all exist.

import numpy as np
import json
from pathlib import Path
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix

# ------------------------------------------------------------------
# 3.1 Build the model
#     Conv2D + MaxPooling2D blocks -- exploits local spatial patterns
#     (edges, textures, opacities) that Dense layers ignore.
# ------------------------------------------------------------------
cnn_model = keras.Sequential([
    keras.layers.Input(shape=(150, 150, 3)),

    keras.layers.Conv2D(16, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D(2, 2),

    keras.layers.Conv2D(32, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D(2, 2),

    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D(2, 2),

    keras.layers.Flatten(),
    keras.layers.Dropout(0.5),          # randomly drops 50% of neurons during training,
                                          # reduces overfitting on a small (251-image) dataset
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(num_classes, activation="softmax"),
], name="CNN")

cnn_model.summary()

# ------------------------------------------------------------------
# 3.2 Compile
# ------------------------------------------------------------------
cnn_model.compile(
    loss="categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),   # matches Dense model's stable LR
    metrics=["accuracy"],
)

# ------------------------------------------------------------------
# 3.3 Train
#     EarlyStopping monitors validation accuracy, stops if it hasn't improved
#     in 5 epochs, and rolls back to the BEST epoch's weights (not the last one).
# ------------------------------------------------------------------
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True,
)

cnn_history = cnn_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=30,
    callbacks=[early_stop],
    verbose=2,
)

# Save the REAL training history to disk
Path("Training_history_jason").mkdir(exist_ok=True)   # Create the folder
with open("Training_history_jason/cnn_history.json", "w") as f:
    json.dump(cnn_history.history, f)

# ------------------------------------------------------------------
# 3.4 Evaluate: overall test accuracy
# ------------------------------------------------------------------
test_loss, test_acc = cnn_model.evaluate(test_ds, verbose=0)
print(f"\nCNN model -- Test accuracy: {test_acc:.4f}")

# ------------------------------------------------------------------
# 3.5 Evaluate: full test set, per-class report + confusion matrix
#     A single accuracy number can hide a model doing well on one class
#     and poorly on another -- this checks every test image individually.
# ------------------------------------------------------------------
all_true = []
all_pred = []

for images, labels in test_ds:
    preds = cnn_model.predict(images, verbose=0)
    all_true.extend(labels.numpy().argmax(axis=1))
    all_pred.extend(preds.argmax(axis=1))

all_true = np.array(all_true)
all_pred = np.array(all_pred)

print(f"\nTotal test images checked: {len(all_true)}")

print("\n=== CNN model -- per-class performance ===")
print(classification_report(all_true, all_pred, target_names=class_names))

print("=== CNN model -- confusion matrix ===")
print("Rows = true class, Columns = predicted class")
cm_cnn = confusion_matrix(all_true, all_pred)
print("            ", "  ".join(f"{c[:8]:>8s}" for c in class_names))
for i, row in enumerate(cm_cnn):
    print(f"{class_names[i][:12]:12s}", "  ".join(f"{v:8d}" for v in row))

Model: "CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 18496)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 18496)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │     1,183,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,207,587 (4.61 MB)

 Trainable params: 1,207,587 (4.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
16/16 - 2s - 106ms/step - accuracy: 0.5020 - loss: 1.0195 - val_accuracy: 0.6061 - val_loss: 0.9379
Epoch 2/30
16/16 - 0s - 28ms/step - accuracy: 0.6972 - loss: 0.7813 - val_accuracy: 0.6212 - val_loss: 0.7273
Epoch 3/30
16/16 - 0s - 27ms/step - accuracy: 0.8287 - loss: 0.5598 - val_accuracy: 0.7121 - val_loss: 0.5846
Epoch 4/30
16/16 - 1s - 32ms/step - accuracy: 0.8287 - loss: 0.4541 - val_accuracy: 0.7576 - val_loss: 0.4861
Epoch 5/30
16/16 - 0s - 29ms/step - accuracy: 0.8685 - loss: 0.3162 - val_accuracy: 0.7727 - val_loss: 0.4570
Epoch 6/30
16/16 - 0s - 29ms/step - accuracy: 0.8685 - loss: 0.2894 - val_accuracy: 0.8030 - val_loss: 0.4230
Epoch 7/30
16/16 - 0s - 31ms/step - accuracy: 0.9124 - loss: 0.2427 - val_accuracy: 0.8182 - val_loss: 0.3936
Epoch 8/30
16/16 - 0s - 29ms/step - accuracy: 0.9163 - loss: 0.1998 - val_accuracy: 0.8333 - val_loss: 0.3743
Epoch 9/30
16/16 - 0s - 29ms/step - accuracy: 0.9203 - loss: 0.1893 - val_accuracy: 0.8485 - val_loss: 0.3725
Epoch 10/

2026-09-24 05:37:00.774568: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## CNN Model — Interpretation

- **Test accuracy: 90.91%**, with macro F1 of 0.90 — the strongest result so far,
  clearly ahead of the Dense baseline (84.85%), consistent with convolutional layers
  being better suited to exploiting the spatial structure in chest X-ray images.

- **Training was stable from epoch 1**, with no `nan` losses or collapse — training
  and validation accuracy climbed together throughout, and validation loss decreased
  smoothly and consistently (0.938 → 0.263) across nearly the entire run, with only
  a very mild uptick in the final couple of epochs.

- **Training accuracy never fully saturates to 100%** (peaking around 98%), unlike
  the Dense model which hit 100% by epoch 5. Combined with the smoother validation
  loss curve, this suggests the CNN is fitting the training data less aggressively
  and generalizing more naturally, rather than memorizing.

- **`EarlyStopping` stopped training at epoch 20** (out of a maximum 30), with
  `restore_best_weights=True` restoring the epoch with the best validation accuracy
  rather than keeping the final epoch's weights.

- **Per-class performance is strong and fairly balanced across all three classes**:
  - **Covid**: precision 0.96, recall 0.96 — excellent and symmetric; only 1 of 26
    true Covid cases missed (misclassified as Normal).
  - **Normal**: precision 0.83, recall 0.95 — catches nearly all true Normal cases,
    but at some cost to precision (a few Viral Pneumonia cases get pulled in).
  - **Viral Pneumonia**: precision 0.94, recall 0.80 — the weakest recall of the
    three; 4 of 20 true cases were missed (3 called Normal, 1 called Covid).

- **The confusion matrix shows Viral Pneumonia is the main source of remaining
  error** — 3 of its 4 misclassifications go to Normal, mirroring the same
  Normal/Viral Pneumonia confusion pattern seen in the Dense model, though
  reduced in overall frequency (4 total misclassifications here vs. 6 for Dense
  across effectively the same two classes).

- **Covid classification remains highly reliable** across both models tested so
  far, with very few Covid cases missed or falsely flagged — the main
  classification challenge in this dataset consistently sits at the
  Normal/Viral Pneumonia boundary, not with Covid detection itself.

- **Overall**: the CNN outperforms the Dense baseline on every metric while
  showing a more stable, better-behaved training curve — supporting the
  expectation that convolutional layers, by exploiting local spatial patterns
  rather than treating every pixel independently, are architecturally better
  suited to this image classification task.

In [39]:
# ==============================================================================
# STEP 4: Deep CNN (Regularized) Model — Build, Train, Evaluate
# ==============================================================================
# Requires Step 1 (data pipeline) to have already run: train_ds, test_ds,
# class_names, num_classes must all exist.

import numpy as np
import json
from pathlib import Path
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix

# ------------------------------------------------------------------
# 4.1 Build the model
#     4x Conv2D + BatchNormalization + MaxPooling2D blocks, with heavier
#     Dropout. Tests whether more depth + stronger regularization helps,
#     or whether it's too much for a 251-image training set.
# ------------------------------------------------------------------
deep_cnn_model = keras.Sequential([
    keras.layers.Input(shape=(150, 150, 3)),

    keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2, 2),

    keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2, 2),

    keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2, 2),

    keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2, 2),

    keras.layers.Flatten(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(num_classes, activation="softmax"),
], name="Deep_CNN_Regularized")

deep_cnn_model.summary()

# ------------------------------------------------------------------
# 4.2 Compile
# ------------------------------------------------------------------
deep_cnn_model.compile(
    loss="categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),   # matches other models' stable LR
    metrics=["accuracy"],
)

# ------------------------------------------------------------------
# 4.3 Train
#     EarlyStopping monitors validation accuracy, stops if it hasn't improved
#     in 5 epochs, and rolls back to the BEST epoch's weights (not the last one).
# ------------------------------------------------------------------
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True,
)

deep_cnn_history = deep_cnn_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=30,
    callbacks=[early_stop],
    verbose=2,
)

# Save the REAL training history to disk
Path("Training_history_jason").mkdir(exist_ok=True)   # Create the folder
with open("Training_history_jason/deep_cnn_history.json", "w") as f:
    json.dump(deep_cnn_history.history, f)

# ------------------------------------------------------------------
# 4.4 Evaluate: overall test accuracy
# ------------------------------------------------------------------
test_loss, test_acc = deep_cnn_model.evaluate(test_ds, verbose=0)
print(f"\nDeep CNN model -- Test accuracy: {test_acc:.4f}")

# ------------------------------------------------------------------
# 4.5 Evaluate: full test set, per-class report + confusion matrix
#     A single accuracy number can hide a model doing well on one class
#     and poorly on another -- this checks every test image individually.
# ------------------------------------------------------------------
all_true = []
all_pred = []

for images, labels in test_ds:
    preds = deep_cnn_model.predict(images, verbose=0)
    all_true.extend(labels.numpy().argmax(axis=1))
    all_pred.extend(preds.argmax(axis=1))

all_true = np.array(all_true)
all_pred = np.array(all_pred)

print(f"\nTotal test images checked: {len(all_true)}")

print("\n=== Deep CNN model -- per-class performance ===")
print(classification_report(all_true, all_pred, target_names=class_names))

print("=== Deep CNN model -- confusion matrix ===")
print("Rows = true class, Columns = predicted class")
cm_deep_cnn = confusion_matrix(all_true, all_pred)
print("            ", "  ".join(f"{c[:8]:>8s}" for c in class_names))
for i, row in enumerate(cm_deep_cnn):
    print(f"{class_names[i][:12]:12s}", "  ".join(f"{v:8d}" for v in row))

Model: "Deep_CNN_Regularized"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 150, 150, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 150, 150, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 75, 75, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 75, 75, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 75, 75, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 37, 37, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 37, 37, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 37, 37, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 18, 18, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 18, 18, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 18, 18, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 9, 9, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 10368)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10368)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 128)            │     1,327,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,569,859 (5.99 MB)

 Trainable params: 1,569,155 (5.99 MB)

 Non-trainable params: 704 (2.75 KB)

Epoch 1/30
16/16 - 4s - 268ms/step - accuracy: 0.6892 - loss: 1.8536 - val_accuracy: 0.3030 - val_loss: 1.0681
Epoch 2/30
16/16 - 1s - 82ms/step - accuracy: 0.9044 - loss: 0.4495 - val_accuracy: 0.6061 - val_loss: 1.0837
Epoch 3/30
16/16 - 1s - 73ms/step - accuracy: 0.9124 - loss: 0.5175 - val_accuracy: 0.5455 - val_loss: 1.1623
Epoch 4/30
16/16 - 1s - 72ms/step - accuracy: 0.9044 - loss: 0.3499 - val_accuracy: 0.3939 - val_loss: 1.3815
Epoch 5/30
16/16 - 1s - 72ms/step - accuracy: 0.9602 - loss: 0.1455 - val_accuracy: 0.3939 - val_loss: 1.6228
Epoch 6/30
16/16 - 1s - 70ms/step - accuracy: 0.9641 - loss: 0.1127 - val_accuracy: 0.3939 - val_loss: 2.0255
Epoch 7/30
16/16 - 1s - 90ms/step - accuracy: 0.9641 - loss: 0.1738 - val_accuracy: 0.3939 - val_loss: 2.4916

Deep CNN model -- Test accuracy: 0.6061

Total test images checked: 66

=== Deep CNN model -- per-class performance ===
                 precision    recall  f1-score   support

          Covid       0.61      0.96      0.75    

2026-09-24 05:48:13.066570: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
/Applications/Anaconda/anaconda3/envs/tf_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Applications/Anaconda/anaconda3/envs/tf_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Applications/Anaconda/anaconda3/envs/tf_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning

## Deep CNN (Regularized) Model — Interpretation

- **Test accuracy: 60.61%**, macro F1 of 0.47 — well below both the Dense baseline
  (84.85%) and the CNN (90.91%). Unlike the earlier fully-collapsed version of this
  model (which predicted a single class for every image), this run does distinguish
  between classes to some extent, but has a specific, complete failure on one class.

- **The "Normal" class was never predicted, not even once.** Precision, recall, and
  F1 are all exactly 0.00 for Normal — every one of the 20 true Normal test images
  was misclassified, 11 as Covid and 9 as Viral Pneumonia. This is a partial
  collapse: rather than the model defaulting to a single class for everything, it
  has effectively "given up" on one specific class while still partially
  distinguishing the other two.

- **Training accuracy climbed quickly and looked healthy in isolation** (69% → 96%
  by epoch 5), which on its own would suggest normal, successful training.

- **Validation loss tells the real story, and diverges sharply from training loss
  almost immediately**: training loss dropped from 1.85 to 0.11-0.17 and stayed low,
  while validation loss rose steadily and substantially throughout — 1.07 → 1.08 →
  1.16 → 1.38 → 1.62 → 2.03 → 2.49. This is a textbook overfitting signature: the
  model is fitting the training set well while getting progressively *worse*,
  not better, at generalizing to unseen data, epoch over epoch.

- **`EarlyStopping` (`patience=5`, `monitor="val_accuracy"`) stopped training at
  epoch 7** — val_accuracy plateaued at 0.3939 for epochs 4 through 7 with no
  improvement, correctly triggering early stopping. `restore_best_weights=True`
  would have restored the best validation-accuracy epoch (likely epoch 2, at
  0.6061), consistent with the reported final test accuracy of 0.6061.

- **Likely cause**: the combination of 4 convolutional blocks, `BatchNormalization`
  at every block, and two `Dropout` layers (0.5, 0.3) is a large amount of
  regularization and depth relative to a training set of only 251 images. This
  appears to be pushing the model toward a degenerate solution where distinguishing
  the harder class (Normal, which the CNN model also showed to be its most
  precision-limited class) gets sacrificed rather than learned.

- **Comparison with the CNN model is informative here**: the standard 3-block CNN
  (no BatchNormalization, single Dropout layer, default depth) achieved 90.91%
  accuracy with balanced performance across all three classes on the same data.
  This strongly suggests the added depth and regularization in this model are not
  helping on a dataset this size — more capacity and more aggressive regularization
  are not automatically better, and here they appear to actively hurt performance
  compared to the simpler CNN.

- **This result is reported as-is rather than adjusted or hidden**, consistent with
  treating model failures as informative findings rather than results to be
  massaged. It reinforces a broader takeaway from this comparison: architecture and
  regularization choices need to be matched to dataset size, and "deeper +
  more regularized" is not a safe default assumption, particularly with only a
  few hundred training images.

In [40]:
# ==============================================================================
# STEP 4b: Deep RNN (LSTM-based) Model — Build, Train, Evaluate
# ==============================================================================
# Requires Step 1 (data pipeline) to have already run: train_ds, test_ds,
# class_names, num_classes must all exist.

import numpy as np
import json
from pathlib import Path
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix

# ------------------------------------------------------------------
# 4b.1 Build the model
#     Each 150x150x3 image is reshaped into a sequence of 150 "rows",
#     each row flattened to 150*3=450 features -> treated as a timestep.
#     2x LSTM layers (stacked), with Dropout for regularization.
# ------------------------------------------------------------------
deep_rnn_model = keras.Sequential([
    keras.layers.Input(shape=(150, 150, 3)),
    keras.layers.Reshape((150, 150 * 3)),   # (timesteps, features)

    keras.layers.LSTM(128, return_sequences=True),
    keras.layers.Dropout(0.3),

    keras.layers.LSTM(64),
    keras.layers.Dropout(0.3),

    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(num_classes, activation="softmax"),
], name="Deep_RNN_LSTM")

deep_rnn_model.summary()

# ------------------------------------------------------------------
# 4b.2 Compile
# ------------------------------------------------------------------
deep_rnn_model.compile(
    loss="categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),   # matches other models' stable LR
    metrics=["accuracy"],
)

# ------------------------------------------------------------------
# 4b.3 Train
#     EarlyStopping monitors validation accuracy, stops if it hasn't improved
#     in 5 epochs, and rolls back to the BEST epoch's weights (not the last one).
# ------------------------------------------------------------------
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True,
)

deep_rnn_history = deep_rnn_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=30,
    callbacks=[early_stop],
    verbose=2,
)

# Save the REAL training history to disk
Path("Training_history_jason").mkdir(exist_ok=True)
with open("Training_history_jason/deep_rnn_history.json", "w") as f:
    json.dump(deep_rnn_history.history, f)

# ------------------------------------------------------------------
# 4b.4 Evaluate: overall test accuracy
# ------------------------------------------------------------------
test_loss, test_acc = deep_rnn_model.evaluate(test_ds, verbose=0)
print(f"\nDeep RNN model -- Test accuracy: {test_acc:.4f}")

# ------------------------------------------------------------------
# 4b.5 Evaluate: full test set, per-class report + confusion matrix
# ------------------------------------------------------------------
all_true = []
all_pred = []

for images, labels in test_ds:
    preds = deep_rnn_model.predict(images, verbose=0)
    all_true.extend(labels.numpy().argmax(axis=1))
    all_pred.extend(preds.argmax(axis=1))

all_true = np.array(all_true)
all_pred = np.array(all_pred)

print(f"\nTotal test images checked: {len(all_true)}")

print("\n=== Deep RNN model -- per-class performance ===")
print(classification_report(all_true, all_pred, target_names=class_names))

print("=== Deep RNN model -- confusion matrix ===")
print("Rows = true class, Columns = predicted class")
cm_deep_rnn = confusion_matrix(all_true, all_pred)
print("            ", "  ".join(f"{c[:8]:>8s}" for c in class_names))
for i, row in enumerate(cm_deep_rnn):
    print(f"{class_names[i][:12]:12s}", "  ".join(f"{v:8d}" for v in row))

Model: "Deep_RNN_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape (Reshape)               │ (None, 150, 450)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 150, 128)       │       296,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 150, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 354,563 (1.35 MB)

 Trainable params: 354,563 (1.35 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
16/16 - 3s - 186ms/step - accuracy: 0.4462 - loss: 1.0317 - val_accuracy: 0.5455 - val_loss: 0.9447
Epoch 2/30
16/16 - 1s - 43ms/step - accuracy: 0.6534 - loss: 0.8685 - val_accuracy: 0.5758 - val_loss: 0.8632
Epoch 3/30
16/16 - 1s - 40ms/step - accuracy: 0.6454 - loss: 0.7841 - val_accuracy: 0.5606 - val_loss: 0.8150
Epoch 4/30
16/16 - 1s - 40ms/step - accuracy: 0.6653 - loss: 0.7056 - val_accuracy: 0.6212 - val_loss: 0.7310
Epoch 5/30
16/16 - 1s - 42ms/step - accuracy: 0.7410 - loss: 0.6373 - val_accuracy: 0.6061 - val_loss: 0.7386
Epoch 6/30
16/16 - 1s - 40ms/step - accuracy: 0.7331 - loss: 0.5802 - val_accuracy: 0.6212 - val_loss: 0.7027
Epoch 7/30
16/16 - 1s - 38ms/step - accuracy: 0.7291 - loss: 0.5797 - val_accuracy: 0.6667 - val_loss: 0.6920
Epoch 8/30
16/16 - 1s - 38ms/step - accuracy: 0.7809 - loss: 0.5067 - val_accuracy: 0.6364 - val_loss: 0.7014
Epoch 9/30
16/16 - 1s - 37ms/step - accuracy: 0.7888 - loss: 0.4837 - val_accuracy: 0.6364 - val_loss: 0.6789
Epoch 10/

2026-09-24 05:51:40.852509: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Deep RNN (LSTM-based) Model — Interpretation

- **Test accuracy: 69.70%**, macro F1 of 0.68 — below the Dense baseline (84.85%)
  and the CNN (90.91%), but a meaningful improvement over the Deep CNN
  (Regularized) model's 60.61%. Unlike that model, this RNN does not collapse on
  any single class — all three classes get non-zero precision and recall — but
  performance is uneven across them.

- **Covid is predicted well** (precision 0.82, recall 0.88, F1 0.85), while
  **Normal is the weakest class** (precision 0.59, recall 0.50, F1 0.54). Half of
  the true Normal images (10 of 20) were misclassified — 2 as Covid, 8 as Viral
  Pneumonia. Viral Pneumonia sits in between (F1 0.63), with some confusion in
  both directions against Normal.

- **Training accuracy climbed steadily and looks healthy** (44.6% → 90.8% by
  epoch 16), a smoother and slower climb than the Deep CNN's jump to 96% by
  epoch 5 — consistent with RNNs typically needing more epochs to fit sequential
  data.

- **Validation loss shows mild overfitting, but nowhere near as severe as the
  Deep CNN.** Training loss fell steadily from 1.03 to 0.27, while validation
  loss dropped from 0.94 to a low around 0.65 (epoch 11), then drifted back up
  and oscillated between roughly 0.68 and 0.79 for the remaining epochs. This is
  a gentler divergence than the Deep CNN's runaway climb to 2.49 — the RNN
  plateaus and wobbles rather than degrading monotonically, suggesting it found
  a reasonably stable (if imperfect) generalization point rather than fully
  overfitting.

- **`EarlyStopping` (`patience=5`, `monitor="val_accuracy"`) stopped training at
  epoch 16** — val_accuracy peaked at 0.6970 in epoch 11 and did not improve for
  5 subsequent epochs, correctly triggering the stop. `restore_best_weights=True`
  would have restored the epoch-11 weights, consistent with the reported final
  test accuracy of 0.6970.

- **Likely explanation**: treating image rows as timesteps gives the LSTM access
  to some sequential/positional structure in the image, but discards the 2D
  spatial locality that convolutions exploit directly (nearby pixels in both
  height and width). This is enough to learn a moderately useful representation
  — clearly better than a collapsed model — but weaker than a CNN's inductive
  bias for image data, particularly for the harder class (Normal, which was also
  the weakest class for the Deep CNN, though for a different reason: the Deep
  CNN missed it entirely, while the RNN partially confuses it with Viral
  Pneumonia).

- **Comparison across all three models on this dataset is informative**: the
  simple 3-block CNN reached 90.91% with balanced classes, the Deep CNN
  (Regularized) collapsed to 60.61% by fully failing on Normal, and this RNN
  reaches 69.70% with no class collapse but weaker discrimination on the
  harder classes. This suggests that architecture choice matters more than raw
  depth or parameter count on a dataset this small (251 training images) —
  convolutional inductive bias suits this image classification task better than
  either an over-regularized deeper CNN or a sequence-based RNN reformulation.

- **This result is reported as-is**, consistent with treating each architecture's
  actual behavior — including partial weaknesses like the Normal/Viral Pneumonia
  confusion — as informative rather than something to downplay. It reinforces
  that matching architecture to the structure of the data (spatial for images)
  is more important than model depth or sequence-processing sophistication when
  training data is limited.